In [1]:
# Cell 1 - Import libraries
import osmnx as ox
import networkx as nx
import folium
from geopy.geocoders import Nominatim
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
# Cell 2 - Fetch real hospitals from OpenStreetMap
print("Fetching hospitals in Chennai from OpenStreetMap...")

# Define the city
city = "Chennai, Tamil Nadu, India"

# Fetch hospitals
hospitals = ox.features_from_place(city, tags={'amenity': 'hospital'})

# Keep only useful columns
hospitals = hospitals[['geometry', 'name']].copy()
hospitals = hospitals.dropna(subset=['name'])

# Extract lat/long from geometry
hospitals['lat'] = hospitals.geometry.centroid.y
hospitals['lon'] = hospitals.geometry.centroid.x
hospitals = hospitals.reset_index(drop=True)

print(f"Found {len(hospitals)} hospitals in Chennai!")
print(hospitals[['name', 'lat', 'lon']].head(10))

Fetching hospitals in Chennai from OpenStreetMap...
Found 616 hospitals in Chennai!
                           name        lat        lon
0   Kalaignar Karunanidhi Nagar  13.038649  80.205216
1                      Kasimedu  13.122775  80.296058
2                Amrit Hospital  13.092201  80.279291
3               Vikram Hospital  13.098501  80.279618
4          City Tower Hospitals  13.084380  80.216086
5  Apollo Hospitals, Tondiarpet  13.128937  80.290562
6                K.H.M.Hospital  13.084442  80.213072
7   Sundaram Medical Foundation  13.082170  80.206982
8   Sri Devi Speciality Hospial  13.088798  80.199469
9          Corporation Hospital  13.033662  80.257247


In [3]:
# Cell 3 - Fetch police stations
print("Fetching police stations in Chennai...")

police = ox.features_from_place(city, tags={'amenity': 'police'})
police = police[['geometry', 'name']].copy()
police = police.dropna(subset=['name'])
police['lat'] = police.geometry.centroid.y
police['lon'] = police.geometry.centroid.x
police = police.reset_index(drop=True)

print(f"Found {len(police)} police stations in Chennai!")
print(police[['name', 'lat', 'lon']].head(10))

Fetching police stations in Chennai...
Found 75 police stations in Chennai!
                            name        lat        lon
0        Mambalam Police Station  13.034957  80.229460
1                  Royapettah PS  13.052011  80.263735
2  S-8 Adambakkam Police Station  12.989396  80.201968
3                Teynampet E3 PS  13.048651  80.249488
4                       Adyar J2  12.997814  80.255718
5             Central Station PS  13.081828  80.275507
6            Chennai Central RPF  13.083061  80.273300
7       Koyambedu Police Station  13.069641  80.200459
8                    T15 SRMC PS  13.034199  80.160249
9                    KK Nagar PS  13.041511  80.205442


In [4]:
# Cell 4 - Find nearest resource to accident location
from geopy.distance import geodesic

def find_nearest_resources(accident_lat, accident_lon, severity, 
                            hospitals_df, police_df):
    
    accident_location = (accident_lat, accident_lon)
    
    # Calculate distance to every hospital
    hospitals_df['distance_km'] = hospitals_df.apply(
        lambda row: geodesic(accident_location, 
                            (row['lat'], row['lon'])).km, axis=1)
    
    # Calculate distance to every police station
    police_df['distance_km'] = police_df.apply(
        lambda row: geodesic(accident_location, 
                            (row['lat'], row['lon'])).km, axis=1)
    
    # Get nearest ones
    nearest_hospital = hospitals_df.nsmallest(3, 'distance_km')[['name', 'distance_km']]
    nearest_police = police_df.nsmallest(2, 'distance_km')[['name', 'distance_km']]
    
    print(f"\n🚨 Accident at ({accident_lat}, {accident_lon})")
    print(f"⚠️  Severity: {severity}")
    print(f"\n🏥 Nearest Hospitals:")
    print(nearest_hospital.to_string(index=False))
    print(f"\n🚔 Nearest Police Stations:")
    print(nearest_police.to_string(index=False))
    
    return nearest_hospital, nearest_police

# Test with a real Chennai location (Anna Salai)
find_nearest_resources(13.0569, 80.2425, "High", hospitals, police)


🚨 Accident at (13.0569, 80.2425)
⚠️  Severity: High

🏥 Nearest Hospitals:
                                       name  distance_km
    Meenakshi Hospital & Childrens Hospital     0.198878
The Child Trust Medical Research Foundation     0.372747
                Medindia Hospitals, Chennai     0.424524

🚔 Nearest Police Stations:
             name  distance_km
  Teynampet E3 PS     1.186299
F4 Greams Road PS     1.372704


(                                            name  distance_km
 347      Meenakshi Hospital & Childrens Hospital     0.198878
 573  The Child Trust Medical Research Foundation     0.372747
 308                  Medindia Hospitals, Chennai     0.424524,
                  name  distance_km
 3     Teynampet E3 PS     1.186299
 43  F4 Greams Road PS     1.372704)

In [5]:
# Cell 5 - Visualise on interactive map
accident_lat, accident_lon = 13.0569, 80.2425

m = folium.Map(location=[accident_lat, accident_lon], zoom_start=14)

# Accident location
folium.Marker(
    [accident_lat, accident_lon],
    popup="🚨 Accident Location",
    icon=folium.Icon(color='red', icon='exclamation-sign')
).add_to(m)

# Top 3 hospitals
top_hospitals = hospitals.nsmallest(3, 'distance_km')
for _, row in top_hospitals.iterrows():
    folium.Marker(
        [row['lat'], row['lon']],
        popup=f"🏥 {row['name']} ({row['distance_km']:.2f} km)",
        icon=folium.Icon(color='blue', icon='plus-sign')
    ).add_to(m)

# Top 2 police stations
top_police = police.nsmallest(2, 'distance_km')
for _, row in top_police.iterrows():
    folium.Marker(
        [row['lat'], row['lon']],
        popup=f"🚔 {row['name']} ({row['distance_km']:.2f} km)",
        icon=folium.Icon(color='darkblue', icon='home')
    ).add_to(m)

# Save map
m.save('data/emergency_map.html')
print("Map saved! Open data/emergency_map.html in your browser!")

Map saved! Open data/emergency_map.html in your browser!
